In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids.results import load_rows


In [ ]:
# Configuration
CSV_FILENAME = "../../2D_Results.csv"
DATASET_FILTER = ["GPS_Data_Simplified_2D_Feature_Map.xlsx" ]   # Set to None to ignore dataset filter
MODEL_FILTERS = ["Logistic Regression (SMOTE)", "MLP Classifier (SMOTE)", "Random Forest (SMOTE)", "LSTM Classifier (SMOTE)", "StackedEnsemble_RF (SMOTE)"]   # Use a list for multiple models, or set to None to have no models TOP_N = 5 # Number of top results to display set None to display all
TOP_N = 5 # Number of top results to display, set None to display all

In [ ]:
def filter_results(csv_filename, dataset_filter=None, model_filters=None):
    filtered = []
    for row in load_rows([csv_filename]):
        if dataset_filter:
            if isinstance(dataset_filter, str):
                if row["dataset"] != dataset_filter:
                    continue
            elif isinstance(dataset_filter, list):
                if row["dataset"] not in dataset_filter:
                    continue
        if model_filters and row["model"] not in model_filters:
            continue
        filtered.append(row)
    return filtered

def print_results(results, top_n=5):
    if not results:
        print("No matching results.")
        return
    # Sort by MCC
    try:
        results = sorted(results, key=lambda r: float(r.get("mcc", "0") or 0), reverse=True)
    except ValueError:
        pass  # If conversion fails, leave unsorted
    headers = list(results[0].keys())
    print("\t".join(headers))
    for row in results[:top_n] if top_n else results:
        row_copy = row.copy()
        for col in ["mcc", "balanced_accuracy", "f1_macro", "g_mean", "accuracy"]:
            if col in row_copy:
                try:
                    val = float(row_copy[col])
                    row_copy[col] = f"{val*100:.4f}"
                except ValueError:
                    pass
        for col in ["train_time_sec"]:
            if col in row_copy:
                try:
                    val = float(row_copy[col])
                    row_copy[col] = f"{val:.5f}"
                except ValueError:
                    pass
        print("\t".join(str(row_copy[h]) for h in headers))

if __name__ == "__main__":
    results = filter_results(
        CSV_FILENAME,
        dataset_filter=DATASET_FILTER,
        model_filters=MODEL_FILTERS
    )
    print_results(results, top_n=TOP_N)